# Sparse Matrix Dump Demo

This notebook explores `data/matrix_dump_0001.txt`, where each record is a sparse `256x256` energy matrix.

Key semantics from the report:

- `sample` is the temporal block
- `set_index` distinguishes parallel matrices inside the same `sample`
- multiple matrices can share the same `sample` and timing metadata
- the dump is sparse, so visualizations should preserve the per-matrix structure rather than collapse everything into one dense timeline


In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from particle_viz import (
    animation_to_html,
    animate_samples,
    load_dump,
    plot_active_hits,
    plot_dbscan_comparison,
    plot_matrix,
    plot_sample_grid,
    records_to_dataframe,
)

plt.style.use('seaborn-v0_8-whitegrid')

## Load the dump

The default path is fixed to `data/matrix_dump_0001.txt` for this v1 demo notebook.

In [ ]:
DATA_PATH = ROOT / 'data' / 'matrix_dump_0001.txt'
records = load_dump(DATA_PATH)
df = records_to_dataframe(records)
record_lookup = {(record.sample, record.set_index): record for record in records}
sample_values = sorted(df['sample'].unique())
set_index_values = sorted(df['set_index'].unique())

print(f'Dataset path: {DATA_PATH}')
print(f'Record count: {len(records)}')
print(f'Unique samples: {df["sample"].nunique()}')
print(f'Sample range: {df["sample"].min()}..{df["sample"].max()}')
print(f'Set index values: {set_index_values}')

## Summary plots

These plots mirror the report's high-level structure:

- how many matrices appear in each `sample`
- which `set_index` values are present over time
- how sparse or dense each matrix is through the `nnz` distribution


In [ ]:
counts_per_sample = df.groupby('sample').size()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

axes[0].bar(counts_per_sample.index, counts_per_sample.values, color='#2c7fb8')
axes[0].set_title('Matrices per sample')
axes[0].set_xlabel('sample')
axes[0].set_ylabel('count')

axes[1].scatter(df['sample'], df['set_index'], s=40, c=df['set_index'], cmap='tab10')
axes[1].set_title('set_index presence over sample')
axes[1].set_xlabel('sample')
axes[1].set_ylabel('set_index')
axes[1].set_yticks(set_index_values)

axes[2].hist(df['nnz'], bins=24, color='#41ab5d', edgecolor='white')
axes[2].set_title('nnz distribution')
axes[2].set_xlabel('active pixels (nnz)')
axes[2].set_ylabel('matrices')

plt.show()

In [ ]:
summary = pd.DataFrame(
    {
        'metric': [
            'records',
            'unique_samples',
            'sample_min',
            'sample_max',
            'set_indices',
            'avg_matrices_per_sample',
            'median_nnz',
            'avg_density',
        ],
        'value': [
            len(records),
            int(df['sample'].nunique()),
            int(df['sample'].min()),
            int(df['sample'].max()),
            ', '.join(map(str, set_index_values)),
            round(float(counts_per_sample.mean()), 2),
            round(float(df['nnz'].median()), 1),
            round(float(df['density'].mean()) * 100, 2),
        ],
    }
)
summary

## Single-matrix explorer

Pick one `sample` and one `set_index` to inspect a single sparse matrix.

The left panel shows the dense `256x256` heatmap with zero entries masked. The right panel shows the active hits directly as a scatter plot.

In [ ]:
def show_single_matrix(sample: int, set_index: int):
    record = record_lookup.get((sample, set_index))
    if record is None:
        display(Markdown(f'**Missing matrix for sample={sample}, set_index={set_index}.**'))
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    ax0, image = plot_matrix(record, ax=axes[0], log_scale=True, origin='lower')
    fig.colorbar(image, ax=ax0, shrink=0.86, pad=0.02, label='Energy')

    ax1, scatter = plot_active_hits(record, ax=axes[1])
    if scatter is not None:
        fig.colorbar(scatter, ax=ax1, shrink=0.86, pad=0.02, label='Energy')

    plt.show()

    metadata = pd.DataFrame(
        {
            'field': [
                'sample',
                'set_index',
                'shape',
                'nnz',
                'total_energy',
                'max_energy',
                'acq_unix',
                'hw_t0_ns',
                'hw_t_proc_ns',
            ],
            'value': [
                record.sample,
                record.set_index,
                record.shape,
                record.nnz,
                round(sum(entry.energy for entry in record.entries), 2),
                round(max((entry.energy for entry in record.entries), default=0.0), 2),
                record.acq_unix,
                round(record.hw_t0_ns, 2),
                round(record.hw_t_proc_ns, 2),
            ],
        }
    )
    display(metadata)

widgets.interact(
    show_single_matrix,
    sample=widgets.SelectionSlider(options=sample_values, value=sample_values[0], description='sample'),
    set_index=widgets.Dropdown(options=set_index_values, value=set_index_values[0], description='set_index'),
);

## Per-sample multi-view grid

This preserves the report's interpretation that one temporal block (`sample`) can contain several parallel matrices distinguished by `set_index`. Missing views remain visible as empty panels so the structure stays comparable across time.

In [ ]:
def show_sample_grid(sample: int):
    fig, _ = plot_sample_grid(records, sample)
    plt.show()

widgets.interact(
    show_sample_grid,
    sample=widgets.SelectionSlider(options=sample_values, value=sample_values[0], description='sample'),
);

## Temporal animation across `sample`

The animation is driven by ordered `sample` values, not by flattening records one-by-one. Each frame keeps the fixed `2x3` layout for `set_index=1..6`.

In [ ]:
animation, animation_fig = animate_samples(records, interval_ms=500)
plt.close(animation_fig)
animation_to_html(animation)

In [ ]:
output_path = ROOT / 'outputs' / 'demo_sample_animation.gif'
export_animation, export_fig = animate_samples(records, interval_ms=500, save_path=output_path)
plt.close(export_fig)
print(f'Saved GIF to: {output_path.resolve()}')

## Raw vs DBSCAN comparison

DBSCAN is a clustering baseline here, not a supervised particle classifier. In this notebook it is used only as a simple `xy` clustering layer to separate local hit islands.

Warning: aggregated or unified frames can merge physically independent events into a single cluster, so the notebook keeps the native per-`sample` / per-`set_index` structure as the primary view.

In [ ]:
def show_dbscan(sample: int, set_index: int, eps: float = 1.5, min_samples: int = 3):
    record = record_lookup.get((sample, set_index))
    if record is None:
        display(Markdown(f'**Missing matrix for sample={sample}, set_index={set_index}.**'))
        return
    fig, _, _ = plot_dbscan_comparison(record, eps=eps, min_samples=min_samples)
    plt.show()

widgets.interact(
    show_dbscan,
    sample=widgets.SelectionSlider(options=sample_values, value=sample_values[0], description='sample'),
    set_index=widgets.Dropdown(options=set_index_values, value=set_index_values[0], description='set_index'),
    eps=widgets.FloatSlider(value=1.5, min=0.5, max=4.0, step=0.1, readout_format='.1f', description='eps'),
    min_samples=widgets.IntSlider(value=3, min=1, max=10, step=1, description='min_samples'),
);